# Imports

In [ ]:
import os
import sys
import time
import cv2
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import cm
import matplotlib as mpl
from tqdm import tqdm
from pathlib import Path
from pathml.preprocessing import TissueDetectionHE, LabelWhiteSpaceHE, LabelArtifactTileHE
from pathml.core import HESlide

sys.path.append('/path/to/pancancer_he_classifier')
from helpers import anno as annoHelper
%load_ext autoreload
%autoreload 2


svs_path = Path('/path/to/svs')
print(svs_path.exists())
tiles_path = Path('/path/to/tiles')
print(tiles_path.exists())
infer_path = Path('/path/to/infer')
print(infer_path.exists())
sampleinfo = Path('/path/to/sampleinfo')
heatmap_path = Path('/path/to/heatmap')
print(heatmap_path.exists())
dswsi_path = heatmap_path.joinpath('ds_only')
slide_df = pd.read_csv('/path/to/samples.tsv',sep='\t')
tile_df = pd.read_csv(infer_path.joinpath('infer_H&E_tiles_df.tsv'),
                      sep='\t')  ## user is responseible to update this tsv file name accordingly
print('Finished')


# Define heatmap colormap & save colorbar

In [ ]:
class MplColorHelper:

  def __init__(self, cmap_name, start_val, stop_val):
    self.cmap_name = cmap_name
    self.cmap = plt.get_cmap(cmap_name)
    self.norm = mpl.colors.Normalize(vmin=start_val, vmax=stop_val)
    self.scalarMap = cm.ScalarMappable(norm=self.norm, cmap=self.cmap)

  def get_rgb(self, val):
    return self.scalarMap.to_rgba(val)

#Generate colorbar:
w=2
cmap = 'RdYlBu'
COL = MplColorHelper(cmap, 0, 1)
val=np.arange(0,1,0.05)
im = np.zeros((len(val),w,3))
bar=[]
for i,v in enumerate(val):
    rgb = np.array(COL.get_rgb(1-v)[0:-1]) * 255
    # rgb = rgb.reshape((1,1,3))
    bar.append(rgb)
fig= plt.figure()
ax = fig.add_subplot()
plt.imshow(np.array(bar).reshape(len(val),1,3).astype('uint8'))
plt.xlabel('')
plt.yticks([0,len(val)/2,len(val)])
ax.set_yticklabels([0,0.5,1])
ax.set_xticklabels('')
ax.invert_yaxis()
plt.ylabel('P(Pos)')
for a in 'xy':
    plt.tick_params(
        axis=a,          # changes apply to the x-axis
        which='both',      # both major and minor ticks are affected
        bottom=False,      # ticks along the bottom edge are off
        top=False,         # ticks along the top edge are off
        labelbottom=False)
plt.box(False)
# fn = ds_path.joinpath('colorbar.eps')
fns=[heatmap_path.joinpath('colorbar.eps'),
     heatmap_path.joinpath('colorbar.png')]
for fn in fns:
    plt.savefig(fn)

# Generate downsampled image of WSIs excluding blank regions

In [ ]:
# make downsampled .png images from each WSI:
def ds_img_from_wsi(wsi_fn,nchunks, ds,verbose = False):
    blank_detect=LabelWhiteSpaceHE(label_name='blank',
                                   proportion_threshold=0.99,
                                   )
    art_detect = LabelArtifactTileHE(label_name='artifact')
    wsi = HESlide(str(wsi_fn))
    x,y= wsi.shape
    tile_size= y // nchunks
    tot_tiles = (x//tile_size) * (y//tile_size)
    print(x,y,tile_size,tot_tiles)
    print('original slide shape:',x,y)
    dsx= x//ds
    dsy= y//ds
    print("ds shape:",dsx,dsy)
    print('tile size:', tile_size, 'total tiles:', tot_tiles)
    blank_image = np.zeros((dsx,dsy,3), np.uint8) +255
    blank_tot=0
    img_tot=0
    for i,tile in enumerate(tqdm(wsi.generate_tiles(shape=tile_size,pad=False))):
        blank_detect.apply(tile)
        if tile.labels['blank'] == False:
            if verbose:
                print('Loading tile %d' % i)
            im =np.array(tile.image)
            img_tot+=1
            xx, yy = tile.coords
            imds= cv2.resize(im,
                             (tile_size//ds,tile_size//ds),
                              interpolation=cv2.INTER_CUBIC)
            dsxx= xx//ds
            dsyy= yy//ds

            blank_image[dsxx:(dsxx + imds.shape[1]),dsyy:(dsyy+imds.shape[0])]=imds
        else:
            blank_tot += 1
            if verbose:
                print('Tile %d detected as blank... skipping' % i)
    print('%d images loaded, %d detected as blank' % (img_tot,blank_tot))
    return blank_image


# Sanity check that tile file name (x,y) are correct in validation csv

In [ ]:
ds = 10
df=tile_df.copy()
slide = Path(df.loc[0,'cur_path']).parent.parts[-1]
base_file = slide
bkg_fn = dswsi_path.joinpath('%s_full_ds%dx_min.png' % (base_file, ds))
bkg = cv2.imread(str(bkg_fn))
bkg = cv2.cvtColor(cv2.cvtColor(bkg,cv2.COLOR_RGB2GRAY),cv2.COLOR_GRAY2RGB)
nchunks=50
if bkg_fn.exists():
    wsi_fn = svs_path.joinpath('%s.svs' % base_file)
    ds_svs = ds_img_from_wsi(wsi_fn,nchunks,ds)
fig = plt.figure(figsize=(50,30))
ax = fig.add_subplot(1,1,1)
comb = cv2.addWeighted(bkg,0.3,ds_svs,0.7, 0)
ax.imshow(comb)


# Perform same examination with heatmap:

In [ ]:
cmap = 'RdYlBu'
COL = MplColorHelper(cmap, 0, 1)
ds = 10
fold = 1
cls_types=['Tumor','notTumor']
thresh = 0.5 #0.16
df=tile_df.copy()
df=tile_df.copy()
slide = Path(df.loc[0,'cur_path']).parent.parts[-1]
base_file = slide
bkg_fn = dswsi_path.joinpath('%s_full_ds%dx_min.png' % (base_file, ds))
bkg = cv2.imread(str(bkg_fn))
bkg = cv2.cvtColor(cv2.cvtColor(bkg,cv2.COLOR_RGB2GRAY),cv2.COLOR_GRAY2RGB)
if bkg_fn.exists():
    wsi = HESlide(str(svs_path.joinpath('%s.svs' % base_file)))
    x,y = wsi.shape
    xds= x // ds
    yds= y // ds
    print('original slide shape:',x,y)
    print("ds shape:",xds,yds)
    tile_fns = df.loc[:,'cur_path'].values
    use_p = df.loc[:,'p_pos'].values
    temp_img = np.zeros((xds,yds,3), np.uint8) +255
    for i,fn in enumerate(tqdm(tile_fns)):
        tn,tx,ty,tile_size= annoHelper.parse_tile_fn(fn)
        im =   np.zeros((ds_ts,ds_ts,3)).astype(np.uint8)
        p = use_p[i]
        rgb = np.array(COL.get_rgb(1-p)[0:-1]) * 255
        rgb = rgb.reshape((1,1,3))
        im = im + rgb
        ds_ts = tile_size//ds 
        dsx=tx//ds
        dsy=ty//ds
        imds = cv2.resize(im,(tile_size//ds,tile_size//ds),interpolation=cv2.INTER_CUBIC)
        temp_img[dsy:(dsy+imds.shape[0]),dsx:(dsx + imds.shape[1])]=imds
fig = plt.figure(figsize=(50,30))
ax = fig.add_subplot(1,1,1)
comb = cv2.addWeighted(bkg,0.6,temp_img,0.4, 30)
ax.imshow(comb)
# ax.set_xlim([1200,1800]);
# ax.set_ylim([600,1100]);

